# Automatron Quant

Signal validation, trade approval gates, and trading-code review.

## Contents

1. Constants and thresholds
2. Sample data
3. Alpha audit tools
4. Trade gate tools
5. Code compliance tools
6. Prompt addenda and workflows
7. Sector pack

Build the core module and put the generated package on the path. This cell is
for interactive use only and is dropped from the built module.

In [ ]:
import pathlib
import subprocess
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
subprocess.run([sys.executable, "scripts/build_notebooks.py"], check=True, cwd=ROOT)
sys.path.insert(0, str(ROOT / "automatron_build"))

In [ ]:
from automatron_core import *  # noqa: F401,F403

## 1. Constants and thresholds

Values read from config/sectors.yaml and the rule files.

In [ ]:
import datetime as dt
import functools
import itertools
import json
import math
import pathlib
import re
from typing import Any

import numpy as np
import yaml

SECTOR_ID = "quant"
THRESHOLDS = sector_settings(SECTOR_ID)["thresholds"]

TRADE_APPROVAL_USD = float(THRESHOLDS["trade_approval_usd"])
HIGH_RISK_SEGMENTS = {str(s).upper() for s in THRESHOLDS["high_risk_segments"]}
DSR_PASS = float(THRESHOLDS["dsr_pass"])
PBO_FAIL = float(THRESHOLDS["pbo_fail"])
CSCV_BLOCKS = int(THRESHOLDS["cscv_blocks"])

SAMPLE_DIR = get_settings().data_path / "samples" / SECTOR_ID
RULES_DIR = ROOT / "config" / "rules"

# Daily bars. Sharpe ratios are always reported annualized by this factor and the
# factor is stated alongside, because the same number means different things at
# daily, weekly and monthly frequency.
TRADING_DAYS = 252
ANNUALIZATION = math.sqrt(TRADING_DAYS)

# Euler-Mascheroni constant, used by the deflated Sharpe ratio's expected-maximum term.
EULER_MASCHERONI = 0.5772156649015329

SAMPLE_TICKERS = ("AAPL", "MSFT", "NVDA", "SPY", "TSLA")
SAMPLE_START = dt.date(2018, 1, 2)
SAMPLE_BARS = 2016  # eight years of trading days

# Bumped whenever a generator changes, so stale sample files are rewritten.
SAMPLE_GENERATOR_VERSION = 4


@functools.lru_cache(maxsize=2)
def load_trade_limits() -> dict[str, Any]:
    with (RULES_DIR / "trade_limits.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


@functools.lru_cache(maxsize=2)
def load_code_controls() -> dict[str, Any]:
    with (RULES_DIR / "code_controls.yaml").open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)


def annualize_sharpe(per_period: float) -> float:
    """Daily Sharpe to annualized. Kept in one place so the factor is never implied."""
    return per_period * ANNUALIZATION

## 2. Sample data

ensure_samples(): simulated prices, variant returns matrices,
tickets, a sample book, list entries and two strategy files.

In [ ]:
# Every bundled dataset here is synthetic and generated from a fixed seed. None of
# it is market data: the tickers are real symbols only so the samples read
# naturally, and the paths behind them were simulated. Anything shown from these
# files is illustrative, which is why each writer stamps that into the file.

SYNTHETIC_NOTE = ("Synthetic data generated for this project from a fixed seed. "
                  "Not market data and not a price history of any real security.")

# Per-ticker drift and volatility for the simulated paths, plus how strongly each
# name loads on the shared market factor. Chosen so the sample book has something
# to concentrate in and VaR differs across names.
TICKER_PROFILE = {
    "AAPL": {"start": 42.0, "drift": 0.00075, "vol": 0.0165, "beta": 1.05, "sector": "Technology"},
    "MSFT": {"start": 85.0, "drift": 0.00082, "vol": 0.0155, "beta": 0.98, "sector": "Technology"},
    "NVDA": {"start": 49.0, "drift": 0.00125, "vol": 0.0290, "beta": 1.45, "sector": "Technology"},
    "SPY": {"start": 268.0, "drift": 0.00042, "vol": 0.0098, "beta": 1.00, "sector": "Index"},
    "TSLA": {"start": 21.0, "drift": 0.00110, "vol": 0.0355, "beta": 1.30, "sector": "Consumer"},
}


def _business_days(start: dt.date, count: int) -> list[dt.date]:
    """Weekday calendar. Holidays are not modelled; the samples are synthetic anyway."""
    days: list[dt.date] = []
    day = start
    while len(days) < count:
        if day.weekday() < 5:
            days.append(day)
        day += dt.timedelta(days=1)
    return days


def _write_prices() -> None:
    """Simulate correlated daily closes for the sample tickers.

    A shared market factor drives most of the covariance, so the pre-trade VaR and
    the concentration check see names that move together rather than independently.
    """
    rng = np.random.default_rng(90210)
    dates = _business_days(SAMPLE_START, SAMPLE_BARS)
    market = rng.normal(0.0, 0.0085, size=SAMPLE_BARS)

    closes: dict[str, np.ndarray] = {}
    for ticker, profile in TICKER_PROFILE.items():
        idiosyncratic = rng.normal(0.0, profile["vol"], size=SAMPLE_BARS)
        rets = profile["drift"] + profile["beta"] * market + idiosyncratic
        closes[ticker] = profile["start"] * np.exp(np.cumsum(rets))

    lines = [f"# {SYNTHETIC_NOTE}", "date," + ",".join(TICKER_PROFILE)]
    for i, day in enumerate(dates):
        row = ",".join(f"{closes[t][i]:.4f}" for t in TICKER_PROFILE)
        lines.append(f"{day.isoformat()},{row}")
    (SAMPLE_DIR / "prices_sample.csv").write_text("\n".join(lines) + "\n", encoding="utf-8")


def _variant_names(count: int) -> list[str]:
    """Names that read like a parameter sweep, because that is what they stand for."""
    fast = (3, 5, 8, 10, 13, 15, 20, 25)
    slow = (30, 60, 90, 120, 150)
    combos = [f"sma_{f}_{s}" for s in slow for f in fast]
    return combos[:count]


def _write_returns_matrix(name: str, seed: int, edge_column: int | None,
                          edge_mean: float, note: str) -> None:
    """Write a T x N matrix of daily variant returns.

    Columns are deliberately NOT demeaned. Removing each column's mean would force
    the in-sample and out-of-sample halves into exact anti-correlation, and the
    overfitting probability would then read 1.0 for an arithmetic reason that has
    nothing to do with overfitting.
    """
    rows, cols = 1260, 40
    rng = np.random.default_rng(seed)
    matrix = rng.normal(0.0, 0.011, size=(rows, cols))
    if edge_column is not None:
        matrix[:, edge_column] += edge_mean

    dates = _business_days(dt.date(2021, 1, 4), rows)
    names = _variant_names(cols)
    lines = [f"# {SYNTHETIC_NOTE}", f"# {note}", "date," + ",".join(names)]
    for i, day in enumerate(dates):
        lines.append(f"{day.isoformat()}," + ",".join(f"{v:.8f}" for v in matrix[i]))
    (SAMPLE_DIR / f"{name}.csv").write_text("\n".join(lines) + "\n", encoding="utf-8")


def _write_book() -> None:
    """A small sample book, used only for the concentration check."""
    # Sized so the book is inside every concentration limit before any ticket is
    # applied. A book that already breached would make every ticket look like a
    # breach, and the check would be reporting the fixture rather than the trade.
    positions = [
        ("AAPL", "Technology", 1_450_000.0), ("MSFT", "Technology", 1_400_000.0),
        ("NVDA", "Technology", 1_200_000.0), ("SPY", "Index", 1_400_000.0),
        ("TSLA", "Consumer", 1_100_000.0), ("KO", "Consumer", 1_200_000.0),
        ("WMT", "Consumer", 1_350_000.0), ("HD", "Consumer", 1_250_000.0),
        ("JPM", "Financials", 1_400_000.0), ("BAC", "Financials", 1_350_000.0),
        ("XOM", "Energy", 1_450_000.0), ("CVX", "Energy", 1_300_000.0),
        ("PFE", "Healthcare", 1_250_000.0), ("UNH", "Healthcare", 1_400_000.0),
    ]
    lines = [f"# {SYNTHETIC_NOTE}", "symbol,sector,market_value_usd"]
    lines += [f"{s},{sec},{v:.2f}" for s, sec, v in positions]
    (SAMPLE_DIR / "book.csv").write_text("\n".join(lines) + "\n", encoding="utf-8")


def _write_restricted_list() -> None:
    """Fictional restricted and watch entries, with a reason the brief can quote."""
    rows = [
        ("TSLA", "RESTRICTED", "ALL", "Fictional research embargo pending a published note."),
        ("PFE", "RESTRICTED", "C-2001", "Fictional client-specific conflict."),
        ("XOM", "WATCH", "ALL", "Fictional heightened review for this sector."),
        ("NVDA", "WATCH", "ALL", "Fictional size review above normal participation."),
        ("ACME", "RESTRICTED", "ALL", "Fictional corporate action in progress."),
    ]
    lines = [f"# {SYNTHETIC_NOTE}", "symbol,list_type,client_id,reason"]
    lines += [",".join(r) for r in rows]
    (SAMPLE_DIR / "restricted_list.csv").write_text("\n".join(lines) + "\n", encoding="utf-8")


def _write_tickets() -> None:
    tickets = {
        # Comfortably inside every limit, ordinary segment, not on any list.
        "ticket_small_ok": {
            "ticket_id": "TKT-2026-0117", "instrument": "MSFT", "asset_class": "equity",
            "side": "BUY", "quantity": 120, "limit_price": 402.50, "currency": "USD",
            "client_id": "C-1044", "client_segment": "PROFESSIONAL",
            "strategy_id": "momentum_us_lc",
            "rationale": "Adding to an existing position after a scheduled rebalance.",
            "trader": "sample.trader",
        },
        # Large enough to pass the approval threshold, from a high-risk segment.
        "ticket_large_highrisk": {
            "ticket_id": "TKT-2026-0118", "instrument": "NVDA", "asset_class": "equity",
            "side": "BUY", "quantity": 9500, "limit_price": 141.80, "currency": "USD",
            "client_id": "C-3307", "client_segment": "HIGH_RISK",
            "strategy_id": "discretionary",
            "rationale": "Discretionary request following an earnings move.",
            "trader": "sample.trader",
        },
        # Small, but the instrument is on the restricted list.
        "ticket_restricted": {
            "ticket_id": "TKT-2026-0119", "instrument": "TSLA", "asset_class": "equity",
            "side": "SELL", "quantity": 400, "limit_price": 248.10, "currency": "USD",
            "client_id": "C-1044", "client_segment": "RETAIL",
            "strategy_id": "mean_reversion_intraday",
            "rationale": "Reducing exposure ahead of a volatility event.",
            "trader": "sample.trader",
        },
    }
    for name, ticket in tickets.items():
        ticket["note"] = SYNTHETIC_NOTE
        (SAMPLE_DIR / f"{name}.json").write_text(
            json.dumps(ticket, indent=2) + "\n", encoding="utf-8")


# Stored with a .txt suffix so neither the import system nor pytest collection can
# pick these up as modules. The compliance workflow reads them as text.
CLEAN_STRATEGY = '''"""Sample strategy with the controls a reviewer expects to find."""
import logging
import os

import pandas as pd

LOG = logging.getLogger(__name__)

MAX_POSITION_NOTIONAL = 250_000.0
MAX_ORDERS_PER_SESSION = 50
DRY_RUN = os.environ.get("STRATEGY_DRY_RUN", "1") == "1"
API_KEY = os.environ["BROKER_API_KEY"]


def build_signal(prices: pd.DataFrame, fast: int = 10, slow: int = 50) -> pd.Series:
    """Cross of two moving averages, shifted so today's signal trades tomorrow."""
    fast_ma = prices["close"].rolling(fast).mean()
    slow_ma = prices["close"].rolling(slow).mean()
    raw = (fast_ma > slow_ma).astype(int)
    return raw.shift(1).fillna(0)


def position_size(signal: float, price: float, equity: float) -> int:
    notional = min(equity * 0.02, MAX_POSITION_NOTIONAL)
    return int((notional * signal) // price)


def trading_halted(session_pnl: float, error_count: int) -> bool:
    """Kill switch: stop on a loss threshold or repeated errors."""
    return session_pnl < -0.03 * MAX_POSITION_NOTIONAL or error_count >= 5


def run_session(broker, prices: pd.DataFrame, equity: float) -> int:
    sent = 0
    errors = 0
    session_pnl = 0.0
    signal = build_signal(prices)
    for timestamp, row in prices.iterrows():
        if trading_halted(session_pnl, errors):
            LOG.warning("halted at %s", timestamp)
            break
        if sent >= MAX_ORDERS_PER_SESSION:
            break
        quantity = position_size(float(signal.loc[timestamp]), float(row["close"]), equity)
        if quantity <= 0:
            continue
        if quantity * float(row["close"]) > MAX_POSITION_NOTIONAL:
            LOG.error("size %s above the limit at %s", quantity, timestamp)
            continue
        LOG.info("submitting %s at %s", quantity, timestamp)
        try:
            if DRY_RUN:
                LOG.info("dry run, nothing sent")
            else:
                broker.submit_order(symbol="MSFT", quantity=quantity, timestamp=timestamp)
            sent += 1
        except (ConnectionError, TimeoutError) as exc:
            errors += 1
            LOG.exception("order failed at %s: %s", timestamp, exc)
    return sent


def test_signal_is_shifted():
    frame = pd.DataFrame({"close": [1.0, 2.0, 3.0, 4.0, 5.0] * 20})
    assert build_signal(frame, 2, 4).iloc[0] == 0
'''

RISKY_STRATEGY = '''"""Sample strategy written to fail review. Do not copy this."""
import pandas as pd
import requests

BROKER_TOKEN = "sk_live_EXAMPLE_ONLY_NOT_A_REAL_CREDENTIAL"


def build_signal(prices):
    fast = prices["close"].rolling(5).mean()
    slow = prices["close"].rolling(40).mean()
    # peeks at tomorrow's average before deciding today
    future = prices["close"].shift(-1)
    return ((fast > slow) & (future > prices["close"])).astype(int)


def next_bar_return(prices, i):
    return prices["close"].iloc[i + 1] / prices["close"].iloc[i] - 1.0


def rule_from_config(expression, row):
    return eval(expression)


def run(broker, prices):
    signal = build_signal(prices)
    i = 0
    while True:
        row = prices.iloc[i % len(prices)]
        size = int(1000 * signal.iloc[i % len(signal)])
        if size:
            try:
                broker.submit_order(symbol="TSLA", quantity=size, price=row["close"])
            except:
                pass
        i += 1


def report(prices):
    import datetime
    stamp = datetime.datetime.now()
    payload = {"as_of": stamp.isoformat(), "rows": len(prices)}
    return requests.post("https://example.invalid/report", json=payload, timeout=5)
'''


def ensure_samples(force: bool = False) -> None:
    """Write the bundled quant samples if they are missing or out of date."""
    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
    stamp = SAMPLE_DIR / ".generator_version"
    current = stamp.read_text(encoding="utf-8").strip() if stamp.is_file() else ""
    if not force and current == str(SAMPLE_GENERATOR_VERSION) and \
            (SAMPLE_DIR / "prices_sample.csv").is_file():
        return

    _write_prices()
    # Pure noise: no variant has an edge, so the audit should call it overfit.
    # A single noise draw is a noisy fixture: across independent draws the deflated
    # Sharpe averages about 0.50 and the overfitting probability about 0.49, both with
    # a wide spread. This seed is a chosen draw at the clear end of that spread so the
    # sample demonstrates the failure mode unambiguously rather than landing on the
    # threshold. The statistics themselves are not tuned; only which draw is shipped.
    _write_returns_matrix(
        "returns_noise", seed=28, edge_column=None, edge_mean=0.0,
        note="Forty variants of pure noise. No variant has an edge by construction.")
    # One variant carries a real but modest mean; the rest are noise.
    _write_returns_matrix(
        "returns_planted_edge", seed=1002, edge_column=13, edge_mean=0.00095,
        note="Variant index 13 carries a planted daily mean of 0.00095; the rest are noise.")
    _write_book()
    _write_restricted_list()
    _write_tickets()
    (SAMPLE_DIR / "strategy_clean.py.txt").write_text(CLEAN_STRATEGY, encoding="utf-8")
    (SAMPLE_DIR / "strategy_risky.py.txt").write_text(RISKY_STRATEGY, encoding="utf-8")
    stamp.write_text(str(SAMPLE_GENERATOR_VERSION) + "\n", encoding="utf-8")

## 3. Alpha audit tools

Price loading, the parameter grid with a one-bar signal
shift, performance metrics, walk-forward folds, the deflated Sharpe ratio,
the overfitting probability, and the audit level.

In [ ]:
class FetchPricesArgs(BaseModel):
    tickers: list[str] = Field(default_factory=lambda: list(SAMPLE_TICKERS))
    start: str = Field(default="", description="ISO date; empty means the whole sample.")
    end: str = Field(default="", description="ISO date; empty means the whole sample.")
    use_live: bool = Field(
        default=False,
        description="Try the market data provider first. Off by default so a run is "
                    "reproducible and needs no network.")


class LoadReturnsArgs(BaseModel):
    file_path: str = Field(default="", description="Uploaded variants matrix (date + columns).")
    sample_name: str = Field(default="returns_noise", description="Bundled matrix instead.")


class BacktestGridArgs(BaseModel):
    strategy: str = Field(default="sma_crossover",
                          description="sma_crossover, momentum or mean_reversion_zscore.")
    param_grid: dict[str, list[float]] = Field(default_factory=dict,
                                               description="Empty means the default grid.")
    ticker: str = Field(default="SPY", description="Which sample price series to trade.")
    cost_bps: float = Field(default=5.0, description="Round-trip cost in basis points.")


class MetricsArgs(BaseModel):
    returns_id: str = Field(default="", description="From a grid or matrix load; latest if empty.")
    variant: str = Field(default="", description="Column name; the best Sharpe if empty.")


class WalkForwardArgs(BaseModel):
    strategy: str = Field(default="sma_crossover")
    param_grid: dict[str, list[float]] = Field(default_factory=dict)
    ticker: str = Field(default="SPY")
    cost_bps: float = Field(default=5.0)
    folds: int = Field(default=5, description="Number of sequential train/test folds.")


class DeflatedSharpeArgs(BaseModel):
    returns_id: str = Field(default="")
    variant: str = Field(default="")
    n_trials: int = Field(default=0, description="Zero means the number of variants tested.")


class PboArgs(BaseModel):
    returns_id: str = Field(default="")
    blocks: int = Field(default=CSCV_BLOCKS, description="Even number of CSCV blocks.")


class VerdictArgs(BaseModel):
    dsr: float = Field(default=-1.0, description="Deflated Sharpe; negative means unknown.")
    pbo: float = Field(default=-1.0, description="Overfitting probability; negative is unknown.")
    wf_positive_share: float = Field(
        default=-1.0, description="Share of folds with an out-of-sample Sharpe above zero.")
    returns_id: str = Field(
        default="",
        description="Measure anything not supplied from this matrix; latest if empty.")


# Loaded price frames and returns matrices stay in the process so several tools
# can share one parse, the same way the telemetry tools do.
_PRICE_CACHE: dict[str, Any] = {}
_RETURNS_CACHE: dict[str, dict[str, Any]] = {}


def _latest(cache: dict[str, Any], key: str):
    """The named entry, or the most recent one.

    Falling back to the latest lets a scripted demo step call these tools without
    knowing an id that only exists once the previous step has run.
    """
    if key and key in cache:
        return cache[key]
    if not key and cache:
        return next(reversed(cache.values()))
    return None


def _read_sample_prices():
    import pandas as pd

    ensure_samples()
    path = SAMPLE_DIR / "prices_sample.csv"
    frame = pd.read_csv(path, comment="#", parse_dates=["date"])
    return frame.set_index("date").sort_index()


def _sharpe_moments(returns: np.ndarray) -> tuple[float, float, float, int]:
    """Per-period Sharpe with the skew and non-excess kurtosis the DSR needs.

    Population moments, not sample-corrected: the deflated Sharpe derivation treats
    these as plug-in estimates, and a normal sample gives kurtosis 3 rather than 0.
    """
    series = np.asarray(returns, dtype=float)
    series = series[np.isfinite(series)]
    count = series.size
    if count < 2:
        return 0.0, 0.0, 3.0, count
    sd = series.std(ddof=0)
    if sd == 0.0:
        return 0.0, 0.0, 3.0, count
    standardized = (series - series.mean()) / sd
    return (float(series.mean() / sd), float((standardized**3).mean()),
            float((standardized**4).mean()), count)


def _sharpe_columns(block: np.ndarray) -> np.ndarray:
    """Per-period Sharpe for every column of a T x N slice."""
    sd = block.std(axis=0, ddof=0)
    mean = block.mean(axis=0)
    out = np.zeros_like(mean)
    np.divide(mean, sd, out=out, where=sd > 0)
    return out


def expected_max_sharpe(variance_of_trials: float, n_trials: int) -> float:
    """SR0: the Sharpe the best of N independent noise trials is expected to reach.

    This is the bar a strategy has to clear before its Sharpe means anything. With
    one trial there is nothing to deflate and the bar is zero.
    """
    if n_trials <= 1 or variance_of_trials <= 0.0:
        return 0.0
    from scipy.stats import norm

    gamma = EULER_MASCHERONI
    upper = norm.ppf(1.0 - 1.0 / n_trials)
    lower = norm.ppf(1.0 - 1.0 / (n_trials * math.e))
    return float(math.sqrt(variance_of_trials) * ((1.0 - gamma) * upper + gamma * lower))


# --- strategy signals ------------------------------------------------------------
# Every signal is shifted one bar before it is multiplied by returns, so a position
# is taken on the bar AFTER the information that justified it. Without that shift a
# backtest reports returns nobody could have earned.

def _sma_crossover(close: np.ndarray, fast: float, slow: float) -> np.ndarray:
    fast, slow = int(fast), int(slow)
    if fast >= slow:
        return np.zeros_like(close)
    fast_ma = _moving_average(close, fast)
    slow_ma = _moving_average(close, slow)
    return (fast_ma > slow_ma).astype(float)


def _momentum(close: np.ndarray, lookback: float, threshold: float = 0.0) -> np.ndarray:
    lookback = int(lookback)
    past = np.concatenate([np.full(lookback, np.nan), close[:-lookback]])
    with np.errstate(invalid="ignore", divide="ignore"):
        change = close / past - 1.0
    return np.where(np.isfinite(change) & (change > threshold), 1.0, 0.0)


def _mean_reversion_zscore(close: np.ndarray, window: float, entry: float) -> np.ndarray:
    window = int(window)
    mean = _moving_average(close, window)
    sq = _moving_average(close**2, window)
    var = np.maximum(sq - mean**2, 0.0)
    sd = np.sqrt(var)
    with np.errstate(invalid="ignore", divide="ignore"):
        z = np.where(sd > 0, (close - mean) / sd, 0.0)
    position = np.zeros_like(close)
    position[z < -entry] = 1.0
    position[z > entry] = -1.0
    return position


def _moving_average(values: np.ndarray, window: int) -> np.ndarray:
    """Trailing mean; the first window-1 entries are nan so they cannot trade."""
    if window <= 1:
        return values.astype(float)
    cumulative = np.concatenate([[0.0], np.cumsum(values, dtype=float)])
    out = np.full(values.size, np.nan)
    out[window - 1:] = (cumulative[window:] - cumulative[:-window]) / window
    return out


STRATEGIES = {
    "sma_crossover": {
        "signal": _sma_crossover, "params": ("fast", "slow"),
        "grid": {"fast": [3, 5, 8, 10, 15, 20, 25, 30], "slow": [40, 60, 90, 120, 150]},
    },
    "momentum": {
        "signal": _momentum, "params": ("lookback", "threshold"),
        "grid": {"lookback": [5, 10, 20, 40, 60, 90, 120, 180],
                 "threshold": [-0.02, 0.0, 0.01, 0.03, 0.05]},
    },
    "mean_reversion_zscore": {
        "signal": _mean_reversion_zscore, "params": ("window", "entry"),
        "grid": {"window": [5, 10, 15, 20, 30, 45, 60, 90], "entry": [0.5, 1.0, 1.5, 2.0, 2.5]},
    },
}


def _variant_returns(close: np.ndarray, bar_returns: np.ndarray, signal_fn,
                     values: tuple[float, ...], cost_rate: float) -> tuple[np.ndarray, float]:
    """One variant's net daily returns and its turnover.

    The signal is computed on closes up to and including today, then shifted one bar
    so it applies to tomorrow's return. Costs are charged on the change in position.
    """
    raw = signal_fn(close, *values)
    position = np.concatenate([[0.0], raw[:-1]])
    position = np.nan_to_num(position, nan=0.0)
    traded = np.abs(np.diff(np.concatenate([[0.0], position])))
    net = position * bar_returns - traded * cost_rate
    return net, float(traded.sum())


def _grid_points(strategy: str, param_grid: dict[str, list[float]]):
    spec = STRATEGIES[strategy]
    grid = {k: list(v) for k, v in (param_grid or spec["grid"]).items()}
    names = [p for p in spec["params"] if p in grid]
    if len(names) != len(spec["params"]):
        raise ValueError(f"{strategy} needs a grid over {list(spec['params'])}")
    combos = list(itertools.product(*(grid[n] for n in names)))
    return names, combos


def _store_returns(matrix: np.ndarray, variants: list[str], meta: dict[str, Any]) -> str:
    returns_id = f"ret_{abs(hash((tuple(variants), matrix.shape, matrix[0, 0]))) % 10**8:08d}"
    _RETURNS_CACHE[returns_id] = {"matrix": matrix, "variants": variants, **meta}
    return returns_id


@tool(args_schema=FetchPricesArgs)
def fetch_prices(tickers: list[str] | None = None, start: str = "", end: str = "",
                 use_live: bool = False) -> dict[str, Any]:
    """Load daily closes for the requested tickers and report the coverage obtained.

    Reads the bundled synthetic series by default, so a run is reproducible and needs
    no network. With use_live the market data provider is tried first and the bundled
    series is the fallback; the result always says which source was used.
    """
    import pandas as pd

    wanted = [t.upper() for t in (tickers or list(SAMPLE_TICKERS))]
    source = "bundled_sample"
    frame = None
    fallback_reason = ""

    if use_live:
        try:
            import yfinance

            downloaded = yfinance.download(wanted, start=start or None, end=end or None,
                                           progress=False, auto_adjust=True)
            closes = downloaded["Close"] if "Close" in downloaded else downloaded
            if closes is not None and len(closes):
                frame = pd.DataFrame(closes).dropna(how="all")
                source = "yfinance_adjusted_close"
        except Exception as exc:  # noqa: BLE001 - any provider failure falls back
            fallback_reason = f"{type(exc).__name__}: {exc}"[:160]

    if frame is None:
        frame = _read_sample_prices()

    available = [t for t in wanted if t in frame.columns]
    missing = [t for t in wanted if t not in frame.columns]
    if not available:
        return {"error": f"none of {wanted} are in the price source",
                "available_columns": list(frame.columns)[:20], "tool_version": 1}

    frame = frame[available]
    if start:
        frame = frame[frame.index >= pd.Timestamp(start)]
    if end:
        frame = frame[frame.index <= pd.Timestamp(end)]
    frame = frame.dropna(how="all")

    dataset_id = f"px_{abs(hash((tuple(available), len(frame), source))) % 10**8:08d}"
    _PRICE_CACHE[dataset_id] = frame

    return {
        "dataset_id": dataset_id,
        "source": source,
        "synthetic": source == "bundled_sample",
        "source_note": SYNTHETIC_NOTE if source == "bundled_sample" else "Provider adjusted close.",
        "fallback_reason": fallback_reason,
        "tickers": available,
        "missing_tickers": missing,
        "bars": int(len(frame)),
        "start": str(frame.index[0].date()) if len(frame) else "",
        "end": str(frame.index[-1].date()) if len(frame) else "",
        "tool_version": 1,
    }


@tool(args_schema=LoadReturnsArgs)
def load_returns_matrix(file_path: str = "", sample_name: str = "returns_noise") -> dict[str, Any]:
    """Validate a variants returns matrix and keep it for the audit tools.

    Expects a date column followed by one column per strategy variant, each holding
    that variant's daily return. Returns the id the other tools take.
    """
    import pandas as pd

    if file_path:
        path = pathlib.Path(file_path)
    else:
        ensure_samples()
        path = SAMPLE_DIR / f"{sample_name.removesuffix('.csv') or 'returns_noise'}.csv"
    if not path.is_file():
        return {"error": f"no returns matrix at {path.name}", "tool_version": 1}

    frame = pd.read_csv(path, comment="#")
    date_column = frame.columns[0]
    variants = [c for c in frame.columns[1:]]
    if not variants:
        return {"error": "the matrix has a date column but no variant columns",
                "tool_version": 1}

    values = frame[variants].apply(pd.to_numeric, errors="coerce")
    bad = [c for c in variants if values[c].isna().all()]
    if bad:
        return {"error": f"these columns hold no numeric returns: {bad[:5]}", "tool_version": 1}

    matrix = values.to_numpy(dtype=float)
    matrix = np.nan_to_num(matrix, nan=0.0)
    extreme = float(np.abs(matrix).max()) if matrix.size else 0.0

    returns_id = _store_returns(matrix, variants, {
        "source": path.name, "dates": frame[date_column].astype(str).tolist(),
        "turnover": {}, "cost_bps": None, "strategy": "uploaded",
    })
    sharpes = _sharpe_columns(matrix)
    best = int(np.argmax(sharpes))
    return {
        "returns_id": returns_id, "source": path.name,
        "observations": int(matrix.shape[0]), "variants": int(matrix.shape[1]),
        "variant_names": variants[:12],
        "best_variant": variants[best],
        "best_annualized_sharpe": round(annualize_sharpe(float(sharpes[best])), 4),
        "annualization_factor": f"sqrt({TRADING_DAYS})",
        "largest_absolute_daily_return": round(extreme, 6),
        "warning": ("a daily return above 100% suggests the file is in percent, not decimals"
                    if extreme > 1.0 else ""),
        "tool_version": 1,
    }


@tool(args_schema=BacktestGridArgs)
def run_backtest_grid(strategy: str = "sma_crossover",
                      param_grid: dict[str, list[float]] | None = None,
                      ticker: str = "SPY", cost_bps: float = 5.0) -> dict[str, Any]:
    """Run every parameter combination over one price series and keep the returns.

    Signals are shifted one bar before they earn anything, so no variant trades on
    information from the bar it is reacting to. Costs are charged on position changes.
    """
    if strategy not in STRATEGIES:
        return {"error": f"unknown strategy '{strategy}'; known: {list(STRATEGIES)}",
                "tool_version": 1}
    try:
        names, combos = _grid_points(strategy, param_grid or {})
    except ValueError as exc:
        return {"error": str(exc), "tool_version": 1}
    if not combos:
        return {"error": "the parameter grid is empty", "tool_version": 1}

    frame = _read_sample_prices()
    symbol = ticker.upper()
    if symbol not in frame.columns:
        return {"error": f"no sample prices for '{symbol}'",
                "available": list(frame.columns), "tool_version": 1}

    close = frame[symbol].to_numpy(dtype=float)
    bar_returns = np.concatenate([[0.0], np.diff(close) / close[:-1]])
    cost_rate = cost_bps / 10000.0
    signal_fn = STRATEGIES[strategy]["signal"]

    columns, labels, turnovers = [], [], {}
    for values in combos:
        net, traded = _variant_returns(close, bar_returns, signal_fn, values, cost_rate)
        label = f"{strategy}_" + "_".join(
            f"{n}{_short(v)}" for n, v in zip(names, values, strict=True))
        columns.append(net)
        labels.append(label)
        turnovers[label] = round(traded, 3)

    matrix = np.column_stack(columns)
    returns_id = _store_returns(matrix, labels, {
        "source": f"{strategy} on {symbol}", "dates": [str(d.date()) for d in frame.index],
        "turnover": turnovers, "cost_bps": cost_bps, "strategy": strategy,
    })

    sharpes = _sharpe_columns(matrix)
    order = np.argsort(sharpes)[::-1]
    return {
        "returns_id": returns_id, "strategy": strategy, "ticker": symbol,
        "variants_tested": len(labels), "observations": int(matrix.shape[0]),
        "cost_bps": cost_bps, "annualization_factor": f"sqrt({TRADING_DAYS})",
        "trials_tried": len(labels),
        "top_variants": [
            {"variant": labels[i],
             "annualized_sharpe": round(annualize_sharpe(float(sharpes[i])), 4),
             "turnover": turnovers[labels[i]]}
            for i in order[:5]
        ],
        "median_annualized_sharpe": round(annualize_sharpe(float(np.median(sharpes))), 4),
        "note": "Signals are shifted one bar; costs are charged on position changes.",
        "tool_version": 1,
    }


def _short(value: float) -> str:
    return str(int(value)) if float(value).is_integer() else f"{value:g}".replace(".", "p")


@tool(args_schema=MetricsArgs)
def performance_metrics(returns_id: str = "", variant: str = "") -> dict[str, Any]:
    """Report the standard performance statistics for one variant.

    Sharpe and Sortino are annualized by the square root of the trading-day count,
    which is stated in the result so the number is never ambiguous.
    """
    entry = _latest(_RETURNS_CACHE, returns_id)
    if entry is None:
        return {"error": "no returns matrix loaded; run a grid or load a matrix first",
                "tool_version": 1}

    matrix, variants = entry["matrix"], entry["variants"]
    sharpes = _sharpe_columns(matrix)
    index = variants.index(variant) if variant in variants else int(np.argmax(sharpes))
    series = matrix[:, index]

    per_period, skew, kurt, count = _sharpe_moments(series)
    downside = series[series < 0]
    downside_sd = float(downside.std(ddof=0)) if downside.size else 0.0
    equity = np.cumprod(1.0 + series)
    peak = np.maximum.accumulate(equity)
    drawdown = float((equity / peak - 1.0).min()) if equity.size else 0.0
    years = count / TRADING_DAYS if count else 0.0
    total = float(equity[-1]) if equity.size else 1.0
    annual_return = (total ** (1.0 / years) - 1.0) if years > 0 and total > 0 else float("nan")

    return {
        "returns_id": returns_id or "latest", "variant": variants[index],
        "observations": count, "years": round(years, 2),
        "annualized_return_pct": (None if math.isnan(annual_return)
                                 else round(annual_return * 100, 3)),
        "annualized_volatility_pct": round(float(series.std(ddof=0)) * ANNUALIZATION * 100, 3),
        "annualized_sharpe": round(annualize_sharpe(per_period), 4),
        "annualized_sortino": (round(float(series.mean()) / downside_sd * ANNUALIZATION, 4)
                               if downside_sd > 0 else None),
        "max_drawdown_pct": round(drawdown * 100, 3),
        "hit_rate_pct": round(float((series > 0).mean()) * 100, 2) if count else 0.0,
        "skew": round(skew, 4), "kurtosis_non_excess": round(kurt, 4),
        "turnover": entry.get("turnover", {}).get(variants[index]),
        "cost_bps": entry.get("cost_bps"),
        "annualization_factor": f"sqrt({TRADING_DAYS})",
        "tool_version": 1,
    }


@tool(args_schema=WalkForwardArgs)
def walk_forward(strategy: str = "sma_crossover",
                 param_grid: dict[str, list[float]] | None = None,
                 ticker: str = "SPY", cost_bps: float = 5.0, folds: int = 5) -> dict[str, Any]:
    """Pick the best parameters on each fold's training window and score the next window.

    Each fold chooses its parameters using only data before the window it is scored
    on, so the out-of-sample Sharpe is what that choice would actually have earned.
    """
    if strategy not in STRATEGIES:
        return {"error": f"unknown strategy '{strategy}'", "tool_version": 1}
    try:
        names, combos = _grid_points(strategy, param_grid or {})
    except ValueError as exc:
        return {"error": str(exc), "tool_version": 1}
    if folds < 2:
        return {"error": "walk-forward needs at least two folds", "tool_version": 1}

    frame = _read_sample_prices()
    symbol = ticker.upper()
    if symbol not in frame.columns:
        return {"error": f"no sample prices for '{symbol}'", "tool_version": 1}

    close = frame[symbol].to_numpy(dtype=float)
    bar_returns = np.concatenate([[0.0], np.diff(close) / close[:-1]])
    cost_rate = cost_bps / 10000.0
    signal_fn = STRATEGIES[strategy]["signal"]

    columns = [_variant_returns(close, bar_returns, signal_fn, values, cost_rate)[0]
               for values in combos]
    matrix = np.column_stack(columns)

    total = matrix.shape[0]
    window = total // (folds + 1)
    if window < 40:
        return {"error": f"only {total} bars: too few for {folds} folds", "tool_version": 1}

    results = []
    for fold in range(folds):
        train_end = window * (fold + 1)
        test_end = min(train_end + window, total)
        train, test = matrix[:train_end], matrix[train_end:test_end]
        if test.shape[0] < 20:
            break
        chosen = int(np.argmax(_sharpe_columns(train)))
        oos = float(_sharpe_columns(test)[chosen])
        results.append({
            "fold": fold + 1,
            "train_bars": int(train.shape[0]), "test_bars": int(test.shape[0]),
            "chosen_variant": "_".join(
                f"{n}{_short(v)}" for n, v in zip(names, combos[chosen], strict=True)),
            "in_sample_annualized_sharpe": round(
                annualize_sharpe(float(_sharpe_columns(train)[chosen])), 4),
            "out_of_sample_annualized_sharpe": round(annualize_sharpe(oos), 4),
        })

    if not results:
        return {"error": "no fold had enough test data", "tool_version": 1}

    positive = [r for r in results if r["out_of_sample_annualized_sharpe"] > 0]
    oos_values = [r["out_of_sample_annualized_sharpe"] for r in results]
    return {
        "strategy": strategy, "ticker": symbol, "folds_completed": len(results),
        "variants_per_fold": len(combos), "cost_bps": cost_bps,
        "folds": results,
        "positive_folds": len(positive),
        "positive_share": round(len(positive) / len(results), 4),
        "median_out_of_sample_sharpe": round(float(np.median(oos_values)), 4),
        "mean_out_of_sample_sharpe": round(float(np.mean(oos_values)), 4),
        "annualization_factor": f"sqrt({TRADING_DAYS})",
        "tool_version": 1,
    }


@tool(args_schema=DeflatedSharpeArgs)
def deflated_sharpe(returns_id: str = "", variant: str = "", n_trials: int = 0) -> dict[str, Any]:
    """Deflate a variant's Sharpe by how many variants were tried to find it.

    Searching a grid produces a best Sharpe even when nothing has an edge. The
    deflated value is the probability the true Sharpe is above zero once that search
    is accounted for, so it falls as the number of trials rises.
    """
    from scipy.stats import norm

    entry = _latest(_RETURNS_CACHE, returns_id)
    if entry is None:
        return {"error": "no returns matrix loaded", "tool_version": 1}

    matrix, variants = entry["matrix"], entry["variants"]
    sharpes = _sharpe_columns(matrix)
    index = variants.index(variant) if variant in variants else int(np.argmax(sharpes))
    per_period, skew, kurt, count = _sharpe_moments(matrix[:, index])

    trials = int(n_trials) if n_trials and n_trials > 0 else len(variants)
    trial_variance = float(np.var(sharpes, ddof=1)) if len(sharpes) > 1 else 0.0
    sr_zero = expected_max_sharpe(trial_variance, trials)

    denominator = 1.0 - skew * per_period + ((kurt - 1.0) / 4.0) * per_period**2
    if denominator <= 0.0 or count < 2:
        return {"error": "the return distribution gives no usable deflation denominator",
                "variant": variants[index], "tool_version": 1}

    statistic = (per_period - sr_zero) * math.sqrt(count - 1) / math.sqrt(denominator)
    value = float(norm.cdf(statistic))
    return {
        "variant": variants[index], "deflated_sharpe": round(value, 6),
        "observed_annualized_sharpe": round(annualize_sharpe(per_period), 4),
        "expected_max_annualized_sharpe_under_no_skill": round(annualize_sharpe(sr_zero), 4),
        "trials_used": trials,
        "variance_of_trial_sharpes": round(trial_variance, 8),
        "observations": count, "skew": round(skew, 4), "kurtosis_non_excess": round(kurt, 4),
        "passes_threshold": bool(value >= DSR_PASS), "threshold": DSR_PASS,
        "annualization_factor": f"sqrt({TRADING_DAYS})",
        "tool_version": 1,
    }


@tool(args_schema=PboArgs)
def pbo_cscv(returns_id: str = "", blocks: int = CSCV_BLOCKS) -> dict[str, Any]:
    """Probability that the in-sample best variant lands below median out of sample.

    The sample is cut into blocks and every balanced way of splitting them becomes one
    in-sample and one out-of-sample set. The variant that wins in sample is ranked out
    of sample; if winning carries no information that rank is uniform, and about half
    the splits fall below the median.
    """
    entry = _latest(_RETURNS_CACHE, returns_id)
    if entry is None:
        return {"error": "no returns matrix loaded", "tool_version": 1}

    matrix, variants = entry["matrix"], entry["variants"]
    count = len(variants)
    if count < 2:
        return {"error": "the overfitting check needs at least two variants", "tool_version": 1}
    if blocks % 2 != 0 or blocks < 4:
        return {"error": "blocks must be an even number of at least four", "tool_version": 1}
    usable = (matrix.shape[0] // blocks) * blocks
    if usable < blocks * 10:
        return {"error": f"only {matrix.shape[0]} observations: too few for {blocks} blocks",
                "tool_version": 1}

    parts = np.array_split(matrix[:usable], blocks, axis=0)
    logits, in_sample_perf, out_sample_perf, chosen = [], [], [], []
    indices = range(blocks)
    for combination in itertools.combinations(indices, blocks // 2):
        rest = [i for i in indices if i not in combination]
        in_sample = np.concatenate([parts[i] for i in combination], axis=0)
        out_sample = np.concatenate([parts[i] for i in rest], axis=0)

        in_sharpe = _sharpe_columns(in_sample)
        out_sharpe = _sharpe_columns(out_sample)
        best = int(np.argmax(in_sharpe))
        rank = float((out_sharpe <= out_sharpe[best]).sum())
        omega = min(max(rank / (count + 1.0), 1e-9), 1.0 - 1e-9)
        logits.append(math.log(omega / (1.0 - omega)))
        in_sample_perf.append(float(in_sharpe[best]))
        out_sample_perf.append(float(out_sharpe[best]))
        chosen.append(best)

    lam = np.array(logits)
    slope = float(np.polyfit(in_sample_perf, out_sample_perf, 1)[0]) if len(lam) > 1 else 0.0
    winners = {variants[i] for i in chosen}
    value = float((lam < 0).mean())
    return {
        "pbo": round(value, 6), "splits_evaluated": len(logits), "blocks": blocks,
        "variants": count,
        "logit_median": round(float(np.median(lam)), 4),
        "performance_degradation_slope": round(slope, 4),
        "median_out_of_sample_annualized_sharpe": round(
            annualize_sharpe(float(np.median(out_sample_perf))), 4),
        "distinct_in_sample_winners": len(winners),
        "exceeds_threshold": bool(value > PBO_FAIL), "threshold": PBO_FAIL,
        "note": ("A slope below one means in-sample performance overstates what follows. "
                 "This is one draw of a noisy statistic, so it is read alongside the "
                 "deflated Sharpe and the walk-forward folds, not on its own."),
        "annualization_factor": f"sqrt({TRADING_DAYS})",
        "tool_version": 1,
    }


@tool(args_schema=VerdictArgs)
def verdict(dsr: float = -1.0, pbo: float = -1.0, wf_positive_share: float = -1.0,
            returns_id: str = "") -> dict[str, Any]:
    """Combine the three statistics into the audit level, by fixed rule.

    Anything not supplied is measured from the loaded returns matrix rather than
    assumed, so the level always rests on numbers this run actually produced. The
    thresholds are the firm's policy and are reported alongside the level.
    """
    measured: list[str] = []
    if (dsr < 0 or pbo < 0) and _latest(_RETURNS_CACHE, returns_id) is not None:
        if dsr < 0:
            computed = deflated_sharpe.invoke({"returns_id": returns_id})
            if "deflated_sharpe" in computed:
                dsr = float(computed["deflated_sharpe"])
                measured.append("deflated Sharpe")
        if pbo < 0:
            computed = pbo_cscv.invoke({"returns_id": returns_id})
            if "pbo" in computed:
                pbo = float(computed["pbo"])
                measured.append("overfitting probability")

    reasons, missing = [], []
    if dsr < 0:
        missing.append("deflated Sharpe")
    if pbo < 0:
        missing.append("overfitting probability")
    if wf_positive_share < 0:
        missing.append("walk-forward fold share")

    overfit = (pbo >= 0 and pbo > PBO_FAIL) or (0 <= dsr < 0.5)
    if pbo >= 0 and pbo > PBO_FAIL:
        reasons.append(f"overfitting probability {pbo:.2f} is above the {PBO_FAIL:.2f} limit")
    if 0 <= dsr < 0.5:
        reasons.append(f"deflated Sharpe {dsr:.2f} is below 0.50")

    if overfit:
        level = "LIKELY_OVERFIT"
    elif (dsr >= DSR_PASS and 0 <= pbo <= 0.2 and wf_positive_share >= 0.6):
        level = "EVIDENCE_OF_EDGE"
        reasons.append(f"deflated Sharpe {dsr:.2f} clears {DSR_PASS:.2f}, overfitting "
                       f"probability {pbo:.2f} is at or below 0.20, and "
                       f"{wf_positive_share * 100:.0f}% of folds were positive out of sample")
    else:
        level = "INCONCLUSIVE"
        if not missing:
            reasons.append("the statistics neither clear the evidence bar nor fail the "
                           "overfitting one")

    if missing:
        reasons.append("not measured: " + ", ".join(missing))
    return {
        "level": level, "reasons": reasons,
        "measured_here": measured,
        "dsr": None if dsr < 0 else round(dsr, 6),
        "pbo": None if pbo < 0 else round(pbo, 6),
        "walk_forward_positive_share": (None if wf_positive_share < 0
                                       else round(wf_positive_share, 4)),
        "thresholds": {"dsr_pass": DSR_PASS, "pbo_fail": PBO_FAIL,
                       "dsr_overfit_below": 0.5, "pbo_evidence_at_or_below": 0.2,
                       "walk_forward_share_at_least": 0.6},
        "tool_version": 1,
    }

## 4. Trade gate tools

Ticket validation, notional at a stated reference rate,
limit and concentration checks, historical value at risk, restricted list
lookup, and the gate level.

In [ ]:
REQUIRED_TICKET_FIELDS = ("ticket_id", "instrument", "asset_class", "side", "quantity",
                          "limit_price", "currency", "client_id", "client_segment",
                          "strategy_id", "trader")
VALID_SIDES = {"BUY", "SELL", "SELL_SHORT", "BUY_TO_COVER"}


class TicketArgs(BaseModel):
    ticket: dict[str, Any] = Field(default_factory=dict, description="Trade ticket fields.")
    sample_name: str = Field(default="", description="Bundled ticket to read instead.")


class CheckLimitsArgs(BaseModel):
    ticket: dict[str, Any] = Field(default_factory=dict)
    notional_usd: float = Field(default=0.0, description="Zero means compute it from the ticket.")
    sample_name: str = Field(default="")


class VarArgs(BaseModel):
    instrument: str = Field(default="", description="Symbol; taken from the sample if empty.")
    notional_usd: float = Field(default=0.0)
    sample_name: str = Field(default="")


class RestrictedArgs(BaseModel):
    instrument: str = Field(default="")
    client_id: str = Field(default="")
    sample_name: str = Field(default="")


class GateArgs(BaseModel):
    notional_usd: float = Field(default=-1.0)
    client_segment: str = Field(default="")
    breaches: list[dict[str, Any]] = Field(default_factory=list)
    restricted: dict[str, Any] = Field(default_factory=dict)
    sample_name: str = Field(default="", description="Evaluate a bundled ticket end to end.")


def _load_ticket(ticket: dict[str, Any], sample_name: str) -> dict[str, Any]:
    """Take the ticket as given, or read a bundled one.

    Tools accept a sample name so each is usable on its own, which is what lets a
    scripted demo step call them without threading state between calls.
    """
    # A named sample identifies the case under review, so it wins over an inline
    # payload. A model that half-remembers the case from an earlier step's text
    # would otherwise redirect the tool onto data it invented.
    if ticket and not sample_name:
        return dict(ticket)
    ensure_samples()
    stem = (sample_name or "ticket_small_ok").removesuffix(".json")
    path = SAMPLE_DIR / f"{stem}.json"
    if not path.is_file():
        raise FileNotFoundError(f"no bundled ticket named '{stem}'")
    return json.loads(path.read_text(encoding="utf-8"))


def _read_csv_rows(name: str) -> list[dict[str, str]]:
    import csv

    ensure_samples()
    path = SAMPLE_DIR / name
    lines = [line for line in path.read_text(encoding="utf-8").splitlines()
             if not line.startswith("#")]
    return list(csv.DictReader(lines))


def _notional_usd(ticket: dict[str, Any]) -> tuple[float, dict[str, Any]]:
    limits = load_trade_limits()
    table = limits["fx_rates_usd"]
    currency = str(ticket.get("currency", "USD")).upper()
    rate = table["rates"].get(currency)
    if rate is None:
        raise ValueError(f"no reference rate configured for '{currency}'")
    local = float(ticket["quantity"]) * float(ticket["limit_price"])
    return local * float(rate), {"currency": currency, "rate_to_usd": float(rate),
                                 "rates_as_of": table["as_of"], "local_notional": local}


@tool(args_schema=TicketArgs)
def validate_ticket(ticket: dict[str, Any] | None = None, sample_name: str = "") -> dict[str, Any]:
    """Check a trade ticket's required fields, side, quantity, price and currency.

    Missing fields are named rather than guessed, because a ticket that is silently
    completed is a ticket nobody reviewed.
    """
    try:
        data = _load_ticket(ticket or {}, sample_name)
    except (FileNotFoundError, ValueError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    limits = load_trade_limits()
    problems: list[str] = []
    missing = [f for f in REQUIRED_TICKET_FIELDS if not str(data.get(f, "")).strip()]
    if missing:
        problems.append(f"missing required fields: {', '.join(missing)}")

    side = str(data.get("side", "")).upper()
    if side and side not in VALID_SIDES:
        problems.append(f"side '{side}' is not one of {sorted(VALID_SIDES)}")

    for field in ("quantity", "limit_price"):
        raw = data.get(field)
        try:
            if raw is not None and float(raw) <= 0:
                problems.append(f"{field} must be greater than zero, got {raw}")
        except (TypeError, ValueError):
            problems.append(f"{field} is not a number: {raw!r}")

    currency = str(data.get("currency", "")).upper()
    if currency and currency not in {c.upper() for c in limits["known_currencies"]}:
        problems.append(f"currency '{currency}' has no configured reference rate")

    asset_class = str(data.get("asset_class", "")).lower()
    if asset_class and asset_class not in limits["asset_class_limits"]:
        problems.append(f"asset class '{asset_class}' has no configured limits")

    return {
        "ticket_id": data.get("ticket_id", ""), "valid": not problems,
        "problems": problems, "fields_present": sorted(k for k in data if k != "note"),
        "instrument": str(data.get("instrument", "")).upper(),
        "client_segment": str(data.get("client_segment", "")).upper(),
        "strategy_id": data.get("strategy_id", ""),
        "tool_version": 1,
    }


@tool(args_schema=TicketArgs)
def compute_notional(ticket: dict[str, Any] | None = None, sample_name: str = "") -> dict[str, Any]:
    """Convert quantity times limit price into USD at the configured reference rate.

    The rate is a fixed value from configuration with the date it was set, never a
    live quote, and both are reported so the number can be checked.
    """
    try:
        data = _load_ticket(ticket or {}, sample_name)
        notional, detail = _notional_usd(data)
    except (FileNotFoundError, ValueError, KeyError, TypeError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    return {
        "ticket_id": data.get("ticket_id", ""),
        "instrument": str(data.get("instrument", "")).upper(),
        "quantity": float(data["quantity"]), "limit_price": float(data["limit_price"]),
        "local_notional": round(detail["local_notional"], 2),
        "currency": detail["currency"], "fx_rate_to_usd": detail["rate_to_usd"],
        "fx_rates_as_of": detail["rates_as_of"],
        "notional_usd": round(notional, 2),
        "approval_threshold_usd": TRADE_APPROVAL_USD,
        "above_approval_threshold": bool(notional >= TRADE_APPROVAL_USD),
        "rate_note": "Fixed reference rate from configuration, not a market quote.",
        "tool_version": 1,
    }


@tool(args_schema=CheckLimitsArgs)
def check_limits(ticket: dict[str, Any] | None = None, notional_usd: float = 0.0,
                 sample_name: str = "") -> dict[str, Any]:
    """Compare a ticket against the instrument, strategy, client and concentration limits.

    Every limit that fires is returned with the value it compared and the threshold it
    compared against, so the brief can show the arithmetic rather than assert a breach.
    """
    try:
        data = _load_ticket(ticket or {}, sample_name)
        notional = float(notional_usd) if notional_usd > 0 else _notional_usd(data)[0]
    except (FileNotFoundError, ValueError, KeyError, TypeError, json.JSONDecodeError) as exc:
        return {"error": str(exc), "tool_version": 1}

    limits = load_trade_limits()
    instrument = str(data.get("instrument", "")).upper()
    asset_class = str(data.get("asset_class", "")).lower()
    segment = str(data.get("client_segment", "")).upper()
    strategy = str(data.get("strategy_id", ""))

    breaches: list[dict[str, Any]] = []
    warnings: list[dict[str, Any]] = []
    checked: list[dict[str, Any]] = []

    def compare(name: str, value: float, cap: float | None, kind: str = "breach") -> None:
        if cap is None:
            return
        entry = {"rule": name, "value_usd": round(value, 2), "limit_usd": round(float(cap), 2),
                 "utilization_pct": round(value / float(cap) * 100, 2) if cap else None}
        checked.append(entry)
        if value > float(cap):
            (breaches if kind == "breach" else warnings).append(entry)

    instrument_cap = (limits["instrument_limits"].get(instrument) or {}).get("max_notional_usd")
    compare(f"instrument_limit:{instrument}", notional, instrument_cap)

    class_cap = (limits["asset_class_limits"].get(asset_class) or {}).get("max_notional_usd")
    compare(f"asset_class_limit:{asset_class}", notional, class_cap)

    strategy_cap = (limits["strategy_limits"].get(strategy) or {}).get("max_ticket_notional_usd")
    compare(f"strategy_limit:{strategy}", notional, strategy_cap)

    segment_cap = (limits["client_segment_limits"].get(segment)
                   or {}).get("max_ticket_notional_usd")
    compare(f"client_segment_limit:{segment}", notional, segment_cap)

    # Concentration is measured against the sample book plus this ticket.
    concentration = limits["concentration"]
    rows = _read_csv_rows("book.csv")
    book_total = sum(float(r["market_value_usd"]) for r in rows)
    existing = sum(float(r["market_value_usd"]) for r in rows if r["symbol"].upper() == instrument)
    sector = next((r["sector"] for r in rows if r["symbol"].upper() == instrument), "")
    sector_existing = sum(float(r["market_value_usd"]) for r in rows if r["sector"] == sector)

    post_total = book_total + notional
    name_share = (existing + notional) / post_total * 100 if post_total else 0.0
    sector_share = (sector_existing + notional) / post_total * 100 if post_total else 0.0

    name_entry = {"rule": f"concentration_single_name:{instrument}",
                  "share_pct": round(name_share, 2),
                  "limit_pct": float(concentration["max_single_name_pct"]),
                  "warn_pct": float(concentration["warn_single_name_pct"]),
                  "book_total_usd": round(book_total, 2)}
    checked.append(name_entry)
    if name_share > float(concentration["max_single_name_pct"]):
        breaches.append(name_entry)
    elif name_share > float(concentration["warn_single_name_pct"]):
        warnings.append(name_entry)

    if sector:
        sector_entry = {"rule": f"concentration_sector:{sector}",
                        "share_pct": round(sector_share, 2),
                        "limit_pct": float(concentration["max_sector_pct"])}
        checked.append(sector_entry)
        if sector_share > float(concentration["max_sector_pct"]):
            breaches.append(sector_entry)

    return {
        "ticket_id": data.get("ticket_id", ""), "instrument": instrument,
        "notional_usd": round(notional, 2),
        "breaches": breaches, "warnings": warnings, "limits_checked": checked,
        "breach_count": len(breaches), "warning_count": len(warnings),
        "book_source": "data/samples/quant/book.csv (synthetic)",
        "tool_version": 1,
    }


@tool(args_schema=VarArgs)
def pre_trade_var(instrument: str = "", notional_usd: float = 0.0,
                  sample_name: str = "") -> dict[str, Any]:
    """One-day historical value at risk for the position this ticket would create.

    The quantile is taken from the instrument's own return history rather than assuming
    a normal distribution, so a fat left tail is carried into the number.
    """
    settings = load_trade_limits()["var"]
    symbol = instrument.upper()
    notional = float(notional_usd)
    if not symbol or notional <= 0:
        try:
            data = _load_ticket({}, sample_name)
            symbol = symbol or str(data.get("instrument", "")).upper()
            notional = notional if notional > 0 else _notional_usd(data)[0]
        except (FileNotFoundError, ValueError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    frame = _read_sample_prices()
    if symbol not in frame.columns:
        return {"error": f"no price history for '{symbol}' in the sample set",
                "available": list(frame.columns), "tool_version": 1}

    closes = frame[symbol].to_numpy(dtype=float)
    lookback = int(settings["lookback_days"])
    window = closes[-(lookback + 1):]
    returns = np.diff(window) / window[:-1]
    if returns.size < int(settings["min_observations"]):
        return {"error": f"only {returns.size} returns available, need "
                         f"{settings['min_observations']}", "tool_version": 1}

    confidence = float(settings["confidence"])
    quantile = float(np.quantile(returns, 1.0 - confidence))
    var_usd = abs(quantile) * notional
    share = var_usd / notional * 100 if notional else 0.0
    warn_at = float(settings["warn_share_of_notional_pct"])

    return {
        "instrument": symbol, "notional_usd": round(notional, 2),
        "confidence": confidence, "horizon_days": int(settings["horizon_days"]),
        "method": "historical simulation on the sample price history",
        "observations_used": int(returns.size),
        "loss_quantile_pct": round(quantile * 100, 4),
        "var_usd": round(var_usd, 2),
        "var_share_of_notional_pct": round(share, 3),
        "above_warning_share": bool(share > warn_at), "warning_share_pct": warn_at,
        "data_note": SYNTHETIC_NOTE,
        "tool_version": 1,
    }


@tool(args_schema=RestrictedArgs)
def restricted_list_check(instrument: str = "", client_id: str = "",
                          sample_name: str = "") -> dict[str, Any]:
    """Look the instrument up on the restricted and watch lists, for this client.

    A restricted entry stops the fast track; a watch entry is reported but does not on
    its own. Entries scoped to one client only apply to that client.
    """
    symbol, client = instrument.upper(), client_id
    if not symbol:
        try:
            data = _load_ticket({}, sample_name)
            symbol = str(data.get("instrument", "")).upper()
            client = client or str(data.get("client_id", ""))
        except (FileNotFoundError, ValueError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    hits = []
    for row in _read_csv_rows("restricted_list.csv"):
        if row["symbol"].upper() != symbol:
            continue
        scope = row["client_id"].strip()
        if scope.upper() != "ALL" and scope != client:
            continue
        hits.append({"symbol": symbol, "list_type": row["list_type"].upper(),
                     "applies_to": scope, "reason": row["reason"]})

    restricted = [h for h in hits if h["list_type"] == "RESTRICTED"]
    return {
        "instrument": symbol, "client_id": client, "hits": hits,
        "restricted": bool(restricted), "watch_only": bool(hits) and not restricted,
        "list_source": "data/samples/quant/restricted_list.csv (fictional entries)",
        "tool_version": 1,
    }


@tool(args_schema=GateArgs)
def gate_decision(notional_usd: float = -1.0, client_segment: str = "",
                  breaches: list[dict[str, Any]] | None = None,
                  restricted: dict[str, Any] | None = None,
                  sample_name: str = "") -> dict[str, Any]:
    """Decide the gate level from the size, the segment, the breaches and the lists.

    Both levels still stop at the approval gate. The fast-track level means nothing
    was triggered, not that the trade may proceed without a person.
    """
    breach_list = list(breaches or [])
    restricted_info = dict(restricted or {})
    notional = float(notional_usd)
    segment = client_segment.upper()

    if sample_name and (notional < 0 or not segment):
        try:
            data = _load_ticket({}, sample_name)
            notional = notional if notional >= 0 else _notional_usd(data)[0]
            segment = segment or str(data.get("client_segment", "")).upper()
            if not breach_list:
                found = check_limits.invoke({"sample_name": sample_name})
                breach_list = found.get("breaches", [])
            if not restricted_info:
                restricted_info = restricted_list_check.invoke({"sample_name": sample_name})
        except (FileNotFoundError, ValueError, KeyError, json.JSONDecodeError) as exc:
            return {"error": str(exc), "tool_version": 1}

    triggers: list[str] = []
    if notional >= TRADE_APPROVAL_USD:
        triggers.append(f"notional {notional:,.2f} USD is at or above the "
                        f"{TRADE_APPROVAL_USD:,.0f} USD approval threshold")
    if segment in HIGH_RISK_SEGMENTS:
        triggers.append(f"client segment {segment} is on the high-risk list")
    for breach in breach_list:
        triggers.append(f"limit breached: {breach.get('rule', 'unnamed rule')}")
    if restricted_info.get("restricted"):
        reasons = "; ".join(h.get("reason", "") for h in restricted_info.get("hits", [])
                            if h.get("list_type") == "RESTRICTED")
        triggers.append(f"instrument is on the restricted list ({reasons})")

    level = "HUMAN_APPROVAL_REQUIRED" if triggers else "WITHIN_LIMITS_FAST_TRACK"
    return {
        "level": level, "triggers": triggers, "trigger_count": len(triggers),
        "notional_usd": round(notional, 2) if notional >= 0 else None,
        "client_segment": segment,
        "thresholds": {"approval_threshold_usd": TRADE_APPROVAL_USD,
                       "high_risk_segments": sorted(HIGH_RISK_SEGMENTS)},
        "note": ("Both levels stop here for sign-off. Nothing in this system places, "
                 "routes, or cancels an order."),
        "tool_version": 1,
    }

## 5. Code compliance tools

Structure extraction and static checks by ast parse,
and the mapping from findings to control themes. Uploaded code is never run.

In [ ]:
# Uploaded code is read as text and parsed with ast. It is never imported, compiled
# to a callable, or executed, and nothing here builds a module from it. That is the
# whole security posture of this workflow: the file is data, not a program.
import ast

# Calls treated as sending an order. Matching by attribute or function name keeps
# this independent of whichever broker library a file happens to use.
ORDER_CALL_NAMES = {
    "submit_order", "place_order", "create_order", "send_order", "new_order",
    "market_order", "limit_order", "submit", "order_target_percent", "order_target",
    "buy", "sell", "execute_order", "place", "order",
}
SIZE_LIMIT_PATTERN = re.compile(
    r"\b(max|limit|cap)_?(position|notional|size|qty|quantity|order|exposure)\w*\b", re.I)
KILL_SWITCH_PATTERN = re.compile(
    r"\b(kill_?switch|circuit_?breaker|trading_?halt\w*|halt(ed)?|panic_?stop|"
    r"should_?stop|emergency_?stop|max_?daily_?loss|loss_?limit|drawdown_?limit)\b", re.I)
DRY_RUN_PATTERN = re.compile(r"\b(dry_?run|paper(_?trad\w*)?|simulat\w*|sandbox|no_?trade)\b", re.I)
SECRET_NAME_PATTERN = re.compile(r"(api_?key|secret|token|password|passwd|credential|private_?key)",
                                 re.I)
SECRET_VALUE_PATTERN = re.compile(
    r"^(sk_(live|test)_[A-Za-z0-9]{12,}|ghp_[A-Za-z0-9]{20,}|AKIA[0-9A-Z]{12,}|"
    r"xox[baprs]-[A-Za-z0-9-]{10,}|AIza[0-9A-Za-z_-]{20,}|gsk_[A-Za-z0-9]{20,})$")
PACING_NAMES = {"sleep", "wait", "throttle", "rate_limit", "pace", "await_rate_limit"}
CLOCK_CALLS = {("datetime", "now"), ("datetime", "utcnow"), ("date", "today"), ("time", "time")}


class ScanArgs(BaseModel):
    file_path: str = Field(default="", description="Path to the uploaded strategy file.")
    sample_name: str = Field(default="strategy_risky",
                             description="Bundled sample to read instead.")
    deployment_target: str = Field(default="live", description="paper or live.")


class StructureArgs(BaseModel):
    file_path: str = Field(default="")
    sample_name: str = Field(default="strategy_risky")


class MapFindingsArgs(BaseModel):
    findings: list[dict[str, Any]] = Field(default_factory=list)
    deployment_target: str = Field(default="live")
    sample_name: str = Field(default="", description="Scan a bundled sample first if given.")


def _read_code(file_path: str, sample_name: str) -> tuple[str, str]:
    """Return the file's text and a display name. Never imports the file."""
    # Unlike the dict payloads a model can invent, a file path names a real upload,
    # so it wins over a bundled sample: shadowing an uploaded file with a sample
    # would scan one file and report on another.
    if file_path:
        path = pathlib.Path(file_path)
    else:
        ensure_samples()
        stem = sample_name or "strategy_risky"
        for suffix in (".py.txt", ".txt", ".py"):
            candidate = SAMPLE_DIR / f"{stem.removesuffix('.py.txt').removesuffix('.txt')}{suffix}"
            if candidate.is_file():
                path = candidate
                break
        else:
            raise FileNotFoundError(f"no bundled strategy sample named '{stem}'")
    if not path.is_file():
        raise FileNotFoundError(f"no file at {path.name}")
    size = path.stat().st_size
    if size > 200 * 1024:
        raise ValueError(f"{path.name} is {size / 1024:.0f} KB, above the 200 KB limit")
    return path.read_text(encoding="utf-8", errors="replace"), path.name


def _is_order_call(node: ast.AST) -> bool:
    if not isinstance(node, ast.Call):
        return False
    func = node.func
    name = func.attr if isinstance(func, ast.Attribute) else getattr(func, "id", "")
    return name in ORDER_CALL_NAMES


def _enclosing_functions(tree: ast.AST) -> list[ast.AST]:
    return [n for n in ast.walk(tree)
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]


def _finding(rule_id: str, line: int, evidence: str, rules: dict[str, Any]) -> dict[str, Any]:
    spec = rules["rules"][rule_id]
    return {"rule_id": rule_id, "title": spec["title"], "theme": spec["theme"],
            "severity": spec["severity"], "line": int(line),
            "evidence": evidence[:200], "detail": " ".join(spec["detail"].split()),
            "remediation": spec["remediation"]}


@tool(args_schema=ScanArgs)
def static_scan(file_path: str = "", sample_name: str = "strategy_risky",
                deployment_target: str = "live") -> dict[str, Any]:
    """Parse a strategy file and report control findings with line numbers.

    The file is parsed with ast and read as text only; it is never imported or run.
    A clean result means these patterns were not found, not that the code is correct.
    """
    try:
        source, display = _read_code(file_path, sample_name)
    except (FileNotFoundError, ValueError) as exc:
        return {"error": str(exc), "tool_version": 1}

    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return {"error": f"{display} does not parse as Python: line {exc.lineno}: {exc.msg}",
                "file": display, "parsed": False, "tool_version": 1}

    rules = load_code_controls()
    lines = source.splitlines()
    findings: list[dict[str, Any]] = []

    def line_text(number: int) -> str:
        return lines[number - 1].strip() if 0 < number <= len(lines) else ""

    # QC-001 credential literals
    for node in ast.walk(tree):
        if not isinstance(node, ast.Assign):
            continue
        for target in node.targets:
            name = getattr(target, "id", "") or getattr(target, "attr", "")
            value = node.value
            if not isinstance(value, ast.Constant) or not isinstance(value.value, str):
                continue
            literal = value.value
            named_secret = SECRET_NAME_PATTERN.search(name) and len(literal) >= 12
            if named_secret or SECRET_VALUE_PATTERN.match(literal):
                findings.append(_finding("QC-001", node.lineno,
                                         f"{name} assigned a literal string", rules))

    # QC-002 look-ahead
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) \
                and node.func.attr == "shift":
            for argument in node.args:
                negative = (isinstance(argument, ast.UnaryOp)
                            and isinstance(argument.op, ast.USub))
                literal_negative = (isinstance(argument, ast.Constant)
                                    and isinstance(argument.value, (int, float))
                                    and argument.value < 0)
                if negative or literal_negative:
                    findings.append(_finding("QC-002", node.lineno,
                                             line_text(node.lineno), rules))
        if isinstance(node, ast.Subscript):
            inner = node.slice
            if isinstance(inner, ast.BinOp) and isinstance(inner.op, ast.Add) \
                    and isinstance(inner.right, ast.Constant) \
                    and isinstance(inner.right.value, int) and inner.right.value > 0:
                findings.append(_finding("QC-002", node.lineno, line_text(node.lineno), rules))

    order_calls = [n for n in ast.walk(tree) if _is_order_call(n)]

    # QC-003 no size limit anywhere near the order path
    if order_calls and not SIZE_LIMIT_PATTERN.search(source):
        findings.append(_finding("QC-003", order_calls[0].lineno,
                                 line_text(order_calls[0].lineno), rules))

    # QC-004 no kill switch
    if order_calls and not KILL_SWITCH_PATTERN.search(source):
        findings.append(_finding("QC-004", order_calls[0].lineno,
                                 line_text(order_calls[0].lineno), rules))

    # QC-005 order call in an unbounded loop with no pacing
    for node in ast.walk(tree):
        if not isinstance(node, ast.While):
            continue
        unbounded = isinstance(node.test, ast.Constant) and bool(node.test.value)
        if not unbounded:
            continue
        body = list(ast.walk(node))
        has_order = any(_is_order_call(n) for n in body)
        paced = any(isinstance(n, ast.Call)
                    and (getattr(n.func, "attr", "") in PACING_NAMES
                         or getattr(n.func, "id", "") in PACING_NAMES)
                    for n in body)
        if has_order and not paced:
            findings.append(_finding("QC-005", node.lineno, "while True with an order call",
                                     rules))

    # QC-006 order calls with no logging in the same function
    for function in _enclosing_functions(tree):
        body = list(ast.walk(function))
        calls = [n for n in body if _is_order_call(n)]
        if not calls:
            continue
        logged = any(
            isinstance(n, ast.Call) and (
                getattr(n.func, "attr", "") in {"info", "warning", "error", "debug",
                                                "exception", "log"}
                or getattr(n.func, "id", "") == "print")
            for n in body)
        if not logged:
            findings.append(_finding("QC-006", calls[0].lineno,
                                     f"in {function.name}()", rules))

    # QC-007 bare except in a function that sends orders
    for function in _enclosing_functions(tree):
        body = list(ast.walk(function))
        if not any(_is_order_call(n) for n in body):
            continue
        for node in body:
            if isinstance(node, ast.ExceptHandler) and node.type is None:
                findings.append(_finding("QC-007", node.lineno,
                                         f"bare except in {function.name}()", rules))

    # QC-008 dynamic execution
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and getattr(node.func, "id", "") in {"eval", "exec"}:
            findings.append(_finding("QC-008", node.lineno, line_text(node.lineno), rules))

    # QC-009 no dry run and no tests
    has_tests = any(f.name.startswith("test_") for f in _enclosing_functions(tree))
    if not has_tests and not DRY_RUN_PATTERN.search(source):
        findings.append(_finding("QC-009", 1, "no dry-run flag and no test functions", rules))

    # QC-010 wall-clock reads
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Attribute):
            continue
        owner = getattr(node.func.value, "id", "") or getattr(node.func.value, "attr", "")
        if (owner, node.func.attr) in CLOCK_CALLS:
            findings.append(_finding("QC-010", node.lineno, line_text(node.lineno), rules))

    findings.sort(key=lambda f: (f["rule_id"], f["line"]))
    deduped: list[dict[str, Any]] = []
    for finding in findings:
        if not any(d["rule_id"] == finding["rule_id"] and d["line"] == finding["line"]
                   for d in deduped):
            deduped.append(finding)

    counts: dict[str, int] = {}
    for finding in deduped:
        counts[finding["severity"]] = counts.get(finding["severity"], 0) + 1

    return {
        "file": display, "parsed": True, "executed": False,
        "analysis": "ast parse and pattern match; the file is never imported or run",
        "lines_scanned": len(lines), "deployment_target": deployment_target,
        "findings": deduped, "finding_count": len(deduped),
        "severity_counts": counts,
        "rules_evaluated": sorted(rules["rules"]),
        "disclaimer": " ".join(rules["disclaimer"].split()),
        "tool_version": 1,
    }


@tool(args_schema=StructureArgs)
def extract_structure(file_path: str = "",
                      sample_name: str = "strategy_risky") -> dict[str, Any]:
    """List a strategy file's functions, classes, imports, order calls and constants.

    Parsed with ast only, so this describes the file without running any of it.
    """
    try:
        source, display = _read_code(file_path, sample_name)
    except (FileNotFoundError, ValueError) as exc:
        return {"error": str(exc), "tool_version": 1}
    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        return {"error": f"{display} does not parse: line {exc.lineno}: {exc.msg}",
                "file": display, "tool_version": 1}

    functions, classes, imports, constants, orders = [], [], [], [], []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            functions.append({"name": node.name, "line": node.lineno,
                              "arguments": [a.arg for a in node.args.args]})
        elif isinstance(node, ast.ClassDef):
            classes.append({"name": node.name, "line": node.lineno})
        elif isinstance(node, ast.Import):
            imports.extend(alias.name for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            imports.append(node.module or "")
        elif _is_order_call(node):
            func = node.func
            name = func.attr if isinstance(func, ast.Attribute) else getattr(func, "id", "")
            orders.append({"call": name, "line": node.lineno})

    for node in tree.body:
        if isinstance(node, ast.Assign) and isinstance(node.value, ast.Constant):
            for target in node.targets:
                name = getattr(target, "id", "")
                if name and name.isupper():
                    value = node.value.value
                    redacted = SECRET_NAME_PATTERN.search(name) and isinstance(value, str)
                    constants.append({"name": name, "line": node.lineno,
                                      "value": "[redacted]" if redacted else value})

    return {
        "file": display, "executed": False,
        "functions": functions, "classes": classes,
        "imports": sorted(set(i for i in imports if i)),
        "module_constants": constants,
        "order_calls": orders, "order_call_count": len(orders),
        "has_tests": any(f["name"].startswith("test_") for f in functions),
        "tool_version": 1,
    }


@tool(args_schema=MapFindingsArgs)
def map_findings_to_controls(findings: list[dict[str, Any]] | None = None,
                             deployment_target: str = "live",
                             sample_name: str = "") -> dict[str, Any]:
    """Group findings under their control themes and decide the review level.

    Whether a finding blocks depends on where the code is going: some are only
    blocking for live deployment, and the level says which target was assumed.
    """
    rules = load_code_controls()
    items = list(findings or [])
    if not items and sample_name:
        scanned = static_scan.invoke({"sample_name": sample_name,
                                      "deployment_target": deployment_target})
        if "error" in scanned:
            return scanned
        items = scanned["findings"]

    target = (deployment_target or "live").lower()
    themes: dict[str, dict[str, Any]] = {}
    blocking, advisory = [], []

    for finding in items:
        rule_id = finding.get("rule_id", "")
        spec = rules["rules"].get(rule_id)
        if spec is None:
            continue
        theme_id = spec["theme"]
        theme = rules["control_themes"][theme_id]
        entry = themes.setdefault(theme_id, {
            "theme": theme["name"], "intent": " ".join(theme["intent"].split()),
            "findings": [], "highest_severity": "low"})
        is_blocking = target in spec.get("blocking", [])
        record = {"rule_id": rule_id, "title": spec["title"], "line": finding.get("line"),
                  "severity": spec["severity"], "blocking_for_target": is_blocking}
        entry["findings"].append(record)
        order = {"low": 0, "medium": 1, "high": 2}
        if order[spec["severity"]] > order[entry["highest_severity"]]:
            entry["highest_severity"] = spec["severity"]
        (blocking if is_blocking else advisory).append(record)

    if blocking:
        level = rules["levels"]["blocking_issues"]
    elif advisory:
        level = rules["levels"]["issues_to_review"]
    else:
        level = rules["levels"]["none_found"]

    return {
        "level": level, "deployment_target": target,
        "themes": list(themes.values()), "theme_count": len(themes),
        "blocking": blocking, "blocking_count": len(blocking),
        "advisory": advisory, "advisory_count": len(advisory),
        "disclaimer": " ".join(rules["disclaimer"].split()),
        "tool_version": 1,
    }

## 6. Prompt addenda and workflows

Sector guidance and the three workflow definitions.

In [ ]:
ADDENDA = {
    "coordinator": (
        "Treat every backtest as guilty of overfitting until the statistics say otherwise. "
        "Ask the human for the economic rationale of the signal: a result with no reason to "
        "exist is a result to distrust however good the numbers look. No orders are ever "
        "placed, routed, or cancelled by this system; the gate exists so a person decides."
    ),
    "analyst": (
        "Report Sharpe ratios annualized and state the annualization factor. State the number "
        "of trials used for deflation. A single overfitting probability is one draw of a noisy "
        "statistic, so report it alongside the deflated Sharpe and the walk-forward folds "
        "rather than leaning on it alone."
    ),
    "researcher": (
        "Statements about validation practice or firm policy must cite a knowledge source. "
        "Anything you cannot source is marked 'needs verification' rather than stated."
    ),
    "executor": (
        "Report the fields a ticket is missing by name rather than inferring them. Uploaded "
        "code is read as text and parsed only; never describe it as having been run."
    ),
}


class AlphaAuditInputs(BaseModel):
    strategy: str = Field(default="sma_crossover",
                          description="sma_crossover, momentum or mean_reversion_zscore.")
    ticker: str = Field(default="SPY", description="Which sample price series to trade.")
    param_grid: dict[str, list[float]] = Field(default_factory=dict,
                                               description="Empty means the default grid.")
    returns_file: str = Field(default="", description="Upload a variants matrix instead.")
    sample_name: str = Field(default="", description="Bundled variants matrix instead.")
    cost_bps: float = Field(default=5.0, description="Round-trip cost in basis points.")
    trials_tried: int = Field(default=0, description="Zero means the grid size.")
    economic_rationale: str = Field(
        default="", description="Why this signal should exist. Absence is itself a finding.")


class TradeGateInputs(BaseModel):
    ticket: dict[str, Any] = Field(default_factory=dict, description="Trade ticket JSON.")
    sample_name: str = Field(default="ticket_large_highrisk", description="Bundled ticket.")


class CodeComplianceInputs(BaseModel):
    file_path: str = Field(default="", description="Uploaded strategy file, read as text.")
    sample_name: str = Field(default="strategy_risky", description="Bundled sample instead.")
    deployment_target: str = Field(default="live", description="paper or live.")


ALPHA_AUDIT_PLAN = Plan(
    objective="Decide whether a backtested edge survives the way it was found.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Load the price history and report the coverage obtained, "
                             "saying plainly whether it is sample or provider data.",
                 tool_hints=["fetch_prices", "load_returns_matrix"],
                 expected_output="dataset coverage"),
        PlanStep(id="s2", agent="analyst",
                 instruction="Run the parameter grid and report the best variant's "
                             "performance, the number of trials, and the costs charged.",
                 tool_hints=["run_backtest_grid", "performance_metrics"], depends_on=["s1"],
                 expected_output="grid results and best-variant metrics"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Deflate the best Sharpe by the number of trials and estimate "
                             "the probability of backtest overfitting.",
                 tool_hints=["deflated_sharpe", "pbo_cscv"], depends_on=["s2"],
                 expected_output="deflated Sharpe and overfitting probability"),
        PlanStep(id="s4", agent="analyst",
                 instruction="Run the walk-forward folds and report how many were positive "
                             "out of sample, then give the audit level.",
                 tool_hints=["walk_forward", "verdict"], depends_on=["s3"],
                 expected_output="fold table and a level"),
        PlanStep(id="s5", agent="researcher",
                 instruction="Find the model-validation practice relevant to this claim, "
                             "including what the firm asks for before paper trading.",
                 tool_hints=["search_knowledge"], expected_output="cited context"),
    ],
)

TRADE_GATE_PLAN = Plan(
    objective="Prepare a trade ticket for a human approval decision.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Validate the ticket and convert it to notional in USD, "
                             "naming any missing field.",
                 tool_hints=["validate_ticket", "compute_notional"],
                 expected_output="validated ticket and notional"),
        PlanStep(id="s2", agent="executor",
                 instruction="Check the instrument against the restricted and watch lists "
                             "for this client.",
                 tool_hints=["restricted_list_check"], depends_on=["s1"],
                 expected_output="list result with reasons"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Compare the ticket against every configured limit and "
                             "estimate the one-day value at risk for the position.",
                 tool_hints=["check_limits", "pre_trade_var"], depends_on=["s1"],
                 expected_output="limit utilization and VaR"),
        PlanStep(id="s4", agent="analyst",
                 instruction="Give the gate level and list every rule that triggered it "
                             "with the threshold it compared against.",
                 tool_hints=["gate_decision"], depends_on=["s2", "s3"],
                 expected_output="a level with triggers"),
        PlanStep(id="s5", agent="researcher",
                 instruction="Find the trading policy that applies to a ticket of this "
                             "size and client segment.",
                 tool_hints=["search_knowledge"], expected_output="cited policy"),
    ],
)

CODE_COMPLIANCE_PLAN = Plan(
    objective="Review a strategy file against the firm's code controls before deployment.",
    steps=[
        PlanStep(id="s1", agent="executor",
                 instruction="Describe the file's structure: functions, imports, order "
                             "calls and configuration constants.",
                 tool_hints=["extract_structure"], expected_output="file structure"),
        PlanStep(id="s2", agent="executor",
                 instruction="Run the static checks and report findings with line numbers.",
                 tool_hints=["static_scan"], depends_on=["s1"],
                 expected_output="findings with line numbers"),
        PlanStep(id="s3", agent="analyst",
                 instruction="Group the findings under their control themes and say which "
                             "block this deployment target.",
                 tool_hints=["map_findings_to_controls"], depends_on=["s2"],
                 expected_output="themes and a level"),
        PlanStep(id="s4", agent="researcher",
                 instruction="Find the control expectations for pre-trade checks, kill "
                             "switches, testing, and record keeping.",
                 tool_hints=["search_knowledge"], expected_output="cited control themes"),
    ],
)

# Demo-mode scripts. Arguments are fixed, which is why every tool accepts a sample
# name or falls back to the dataset loaded most recently.
ALPHA_AUDIT_SCRIPT = {
    "s1": [{"tool_calls": [{"name": "fetch_prices", "args": {"tickers": ["SPY"]}}]}, "loaded"],
    "s2": [{"tool_calls": [
        {"name": "run_backtest_grid",
         "args": {"strategy": "sma_crossover", "ticker": "SPY", "cost_bps": 5.0}},
        {"name": "performance_metrics", "args": {}}]}, "grid run"],
    "s3": [{"tool_calls": [
        {"name": "deflated_sharpe", "args": {}},
        {"name": "pbo_cscv", "args": {}}]}, "deflated"],
    "s4": [{"tool_calls": [
        {"name": "walk_forward", "args": {"strategy": "sma_crossover", "ticker": "SPY"}},
        {"name": "verdict", "args": {}}]}, "judged"],
    "s5": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "backtest overfitting deflated sharpe model validation", "k": 4}}]},
        "context gathered"],
}

TRADE_GATE_SCRIPT = {
    "s1": [{"tool_calls": [
        {"name": "validate_ticket", "args": {"sample_name": "ticket_large_highrisk"}},
        {"name": "compute_notional", "args": {"sample_name": "ticket_large_highrisk"}}]},
        "validated"],
    "s2": [{"tool_calls": [{"name": "restricted_list_check",
                            "args": {"sample_name": "ticket_large_highrisk"}}]}, "checked"],
    "s3": [{"tool_calls": [
        {"name": "check_limits", "args": {"sample_name": "ticket_large_highrisk"}},
        {"name": "pre_trade_var", "args": {"sample_name": "ticket_large_highrisk"}}]},
        "assessed"],
    "s4": [{"tool_calls": [{"name": "gate_decision",
                            "args": {"sample_name": "ticket_large_highrisk"}}]}, "decided"],
    "s5": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "trade approval policy notional threshold high risk client", "k": 4}}]},
        "policy found"],
}

CODE_COMPLIANCE_SCRIPT = {
    "s1": [{"tool_calls": [{"name": "extract_structure",
                            "args": {"sample_name": "strategy_risky"}}]}, "described"],
    "s2": [{"tool_calls": [{"name": "static_scan", "args": {
        "sample_name": "strategy_risky", "deployment_target": "live"}}]}, "scanned"],
    "s3": [{"tool_calls": [{"name": "map_findings_to_controls", "args": {
        "sample_name": "strategy_risky", "deployment_target": "live"}}]}, "mapped"],
    "s4": [{"tool_calls": [{"name": "search_knowledge", "args": {
        "query": "pre-trade controls kill switch logging record keeping", "k": 4}}]},
        "controls found"],
}

ALPHA_AUDIT_WORKFLOW = WorkflowSpec(
    id="quant.alpha_audit",
    name="Alpha audit",
    description="Test whether a backtested edge survives the search that produced it.",
    input_schema=AlphaAuditInputs,
    accepted_uploads=[".csv"],
    step_template="1. load prices  2. run the grid and measure the best variant  "
                  "3. deflate and estimate overfitting  4. walk forward and judge  "
                  "5. gather validation context",
    default_plan=ALPHA_AUDIT_PLAN,
    fake_script={"steps": ALPHA_AUDIT_SCRIPT},
    level_vocab=["LIKELY_OVERFIT", "EVIDENCE_OF_EDGE", "INCONCLUSIVE"],
    forbidden_phrases=[r"\bguaranteed\b", r"\bwill outperform\b",
                       r"\bdeploy to production\b", r"\brisk[- ]free\b"],
    sample_name="returns_noise",
    example_request="Audit this moving-average grid on SPY and tell me whether the edge is real.",
)

TRADE_GATE_WORKFLOW = WorkflowSpec(
    id="quant.trade_gate",
    name="Trade approval gate",
    description="Size a ticket, check it against limits and lists, and prepare it for sign-off.",
    input_schema=TradeGateInputs,
    accepted_uploads=[".json"],
    step_template="1. validate and size  2. check the lists  3. limits and VaR  "
                  "4. gate level  5. gather policy",
    default_plan=TRADE_GATE_PLAN,
    fake_script={"steps": TRADE_GATE_SCRIPT},
    level_vocab=["HUMAN_APPROVAL_REQUIRED", "WITHIN_LIMITS_FAST_TRACK"],
    forbidden_phrases=[r"\border (?:placed|sent|executed|routed|submitted)\b",
                       r"\btrade executed\b", r"\bfilled at\b"],
    sample_name="ticket_large_highrisk",
    example_request="Run this ticket through the gate and tell me what a reviewer needs to see.",
)

CODE_COMPLIANCE_WORKFLOW = WorkflowSpec(
    id="quant.code_compliance",
    name="Trading-code compliance review",
    description="Static review of a strategy file against the firm's code controls.",
    input_schema=CodeComplianceInputs,
    accepted_uploads=[".py", ".txt"],
    step_template="1. describe the structure  2. run the static checks  "
                  "3. map findings to controls  4. gather control context",
    default_plan=CODE_COMPLIANCE_PLAN,
    fake_script={"steps": CODE_COMPLIANCE_SCRIPT},
    level_vocab=["BLOCKING_ISSUES", "ISSUES_TO_REVIEW", "NO_ISSUES_FOUND"],
    forbidden_phrases=[r"\bcode (?:was )?executed\b", r"\bwe ran (?:the|your) (?:code|strategy)\b",
                       r"\bsafe to deploy\b", r"\bfully audited\b"],
    sample_name="strategy_risky",
    example_request="Review this strategy file before we let it near the live environment.",
)

## 7. Sector pack

Tool allowlists per role, and registration.

In [ ]:
SECTOR_PACK = SectorPack(
    **sector_identity(SECTOR_ID),
    tools=[
        ToolSpec(tool=fetch_prices, roles=["executor"]),
        ToolSpec(tool=load_returns_matrix, roles=["executor"]),
        ToolSpec(tool=run_backtest_grid, roles=["analyst"]),
        ToolSpec(tool=performance_metrics, roles=["analyst"]),
        ToolSpec(tool=walk_forward, roles=["analyst"]),
        ToolSpec(tool=deflated_sharpe, roles=["analyst"]),
        ToolSpec(tool=pbo_cscv, roles=["analyst"]),
        ToolSpec(tool=verdict, roles=["analyst"]),
        ToolSpec(tool=validate_ticket, roles=["executor"]),
        ToolSpec(tool=compute_notional, roles=["executor"]),
        ToolSpec(tool=restricted_list_check, roles=["executor"]),
        ToolSpec(tool=check_limits, roles=["analyst"]),
        ToolSpec(tool=pre_trade_var, roles=["analyst"]),
        ToolSpec(tool=gate_decision, roles=["analyst"]),
        ToolSpec(tool=static_scan, roles=["executor"]),
        ToolSpec(tool=extract_structure, roles=["executor"]),
        ToolSpec(tool=map_findings_to_controls, roles=["analyst"]),
    ],
    workflows=[ALPHA_AUDIT_WORKFLOW, TRADE_GATE_WORKFLOW, CODE_COMPLIANCE_WORKFLOW],
    addenda=ADDENDA,
    ensure_samples=ensure_samples,
)

register_sector(SECTOR_PACK)

## Demo

Examples only; this cell is dropped from the built module.

In [ ]:
# Run every workflow offline and show what the reviewer would see.
import asyncio

ensure_samples()
for workflow in SECTOR_PACK.workflows:
    print(f"--- {workflow.id} ---")
    run_id = asyncio.run(start_run("quant", workflow.id, workflow.example_request, {}))
    view = asyncio.run(get_run(run_id))
    print(view.status, view.brief["recommendation_level"] if view.brief else "no brief")

## Build check

Confirms the notebook reached the generated module.

In [ ]:
def hello() -> str:
    """Return this module's name, so the build pipeline can be checked end to end."""
    return "automatron_quant"